# CS 521 HW3, Problem 1: Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

Based on the starter notebook of Problem 3. Both models are trained from scratch with `make_model()` for 5 epochs with learning rate 0.01:
a standard model and an $\ell_\infty$ PGD adversarially trained model ($\epsilon = 8/255$).

In [ ]:
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [2]:
def load_data(path):
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [3]:
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)

# The Attack

$\ell_\infty$ PGD from HW1. FGSM is one step of PGD with step size $\epsilon$ and no random start.

In [4]:
def clip_box(x, delta):
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def fgsm(model, x, y, eps):
    return pgd_linf(model, x, y, eps, eps, 1, rand_init=False)

# Training

`train(model, loss_fn)` trains `model` for 5 epochs on the 10,000 training images. The loss function makes the
adversarial examples itself, with the model in `eval` mode, and calculates the loss in `train` mode.

In [5]:
def train(model, loss_fn, epochs=5, lr=0.01, batch_size=128, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:
                continue
            loss = loss_fn(model, augment(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model


def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


def linf_loss(model, x, y, eps=8 / 255, steps=10):
    model.eval()
    x_adv = pgd_linf(model, x, y, eps, 2.5 * eps / steps, steps)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [6]:
model_std = train(make_model(), standard_loss)

epoch 1: loss 1.863 (1 s)


epoch 2: loss 1.526 (2 s)


epoch 3: loss 1.360 (3 s)


epoch 4: loss 1.210 (4 s)


epoch 5: loss 1.118 (5 s)


In [7]:
model_adv = train(make_model(), linf_loss)

epoch 1: loss 2.183 (5 s)


epoch 2: loss 2.097 (10 s)


epoch 3: loss 2.057 (15 s)


epoch 4: loss 2.028 (19 s)


epoch 5: loss 2.020 (24 s)


# Evaluation

`accuracy(model, attack, eps)` is the accuracy (in %) on the 1,000 test images under the attack.
With `eps = 0` it is the standard (clean) accuracy.

In [8]:
def accuracy(model, attack, eps, batch_size=250):
    model.eval()
    torch.manual_seed(0)
    correct = 0
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        x_adv = attack(model, x, y, eps)
        with torch.no_grad():
            correct += (model(x_adv).argmax(1) == y).sum().item()
    return 100.0 * correct / len(y_test)


def pgd(model, x, y, eps, steps=20):
    return pgd_linf(model, x, y, eps, 2.5 * eps / steps, steps)


EPSILONS = [0, 1 / 255, 2 / 255, 4 / 255, 8 / 255, 12 / 255, 16 / 255]

# Part (a): Standard and Robust Accuracy (PGD)

In [9]:
print(f"{'eps':>8}  {'standard':>9}  {'adv-trained':>11}")
for eps in EPSILONS:
    print(f"{eps * 255:5.0f}/255  {accuracy(model_std, pgd, eps):9.1f}  {accuracy(model_adv, pgd, eps):11.1f}")

     eps   standard  adv-trained


    0/255       60.3         37.2


    1/255       37.2         36.0


    2/255       21.5         33.8


    4/255        5.4         30.6


    8/255        0.3         25.4


   12/255        0.0         18.4


   16/255        0.0         13.3


# Part (b): PGD vs. FGSM

In [10]:
print(f"{'eps':>8}  {'std PGD':>8}  {'std FGSM':>8}  {'adv PGD':>8}  {'adv FGSM':>8}")
for eps in EPSILONS:
    row = [accuracy(m, a, eps) for m in (model_std, model_adv) for a in (pgd, fgsm)]
    print(f"{eps * 255:5.0f}/255  " + "  ".join(f"{v:8.1f}" for v in row))

     eps   std PGD  std FGSM   adv PGD  adv FGSM


    0/255      60.3      60.3      37.2      37.2


    1/255      37.2      39.1      36.0      36.0


    2/255      21.5      26.9      33.8      33.8


    4/255       5.4      11.3      30.6      30.7


    8/255       0.3       3.4      25.4      25.9


   12/255       0.0       1.5      18.4      19.7


   16/255       0.0       0.8      13.3      16.0
